In [2]:
import dspy
print(dspy.__version__)

3.0.0


In [ ]:
import retrieve_dspy

query_writer = retrieve_dspy.QueryExpander(
    collection_name="FreshstackLangchain",
    target_property_name="docs_text",
    retrieved_k=20,
    verbose=False
)

query_writer("How can I use Weaviate with LangChain?")

In [21]:
import os

import weaviate

from retrieve_dspy.metrics import create_metric, create_coverage_metric_with_feedback
from retrieve_dspy.datasets.in_memory import load_queries_in_memory

trainset, testset = load_queries_in_memory(
    dataset_name="freshstack-langchain",
    train_samples=30,
    test_samples=20
)

weaviate_client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.getenv("WEAVIATE_URL"),
    auth_credentials=weaviate.auth.AuthApiKey(os.getenv("WEAVIATE_API_KEY")),
)

metric_for_gepa = create_coverage_metric_with_feedback(
    weaviate_client=weaviate_client,
    dataset_name="freshstack-langchain"
)

evaluator = retrieve_dspy.utils.get_evaluator(
    testset=testset,
    metric=metric_for_gepa
)

retrieve_dspy.utils.save_training_questions(trainset, "gepa_query_expander_training_samples.jsonl")

/Users/cshorten/Desktop/retrieve-dspy/.venv/lib/python3.11/site-packages/datasets/utils/py_utils.py:335: ResourceWarning: unclosed <ssl.SSLSocket fd=153, family=2, type=1, proto=0, laddr=('10.0.0.233', 59877), raddr=('172.66.0.243', 443)>
  yield key, tuple(d[key] for d in dicts)
/Users/cshorten/Desktop/retrieve-dspy/.venv/lib/python3.11/site-packages/datasets/utils/py_utils.py:335: ResourceWarning: unclosed <ssl.SSLSocket fd=154, family=2, type=1, proto=0, laddr=('10.0.0.233', 59897), raddr=('162.159.140.245', 443)>
  yield key, tuple(d[key] for d in dicts)
/Users/cshorten/Desktop/retrieve-dspy/.venv/lib/python3.11/site-packages/datasets/utils/py_utils.py:335: ResourceWarning: unclosed <ssl.SSLSocket fd=134, family=2, type=1, proto=0, laddr=('10.0.0.233', 59898), raddr=('162.159.140.245', 443)>
  yield key, tuple(d[key] for d in dicts)
/Users/cshorten/Desktop/retrieve-dspy/.venv/lib/python3.11/site-packages/datasets/utils/py_utils.py:335: ResourceWarning: unclosed <ssl.SSLSocket fd=15

{'path': 'gepa_query_expander_training_samples.jsonl',
 'added': 30,
 'total_in_file': 30}

In [22]:
trainset[0]

Example({'question': 'I have been reading the documentation all day and can\'t seem to wrap my head around how I can create a VectorStoreIndex with llama_index and use the created embeddings as supplemental information for a RAG application/chatbot that can communicate with a user. I want to use llama_index because they have some cool ways to perform more advanced retrieval techniques like sentence window retrieval and auto-merging retrieval (to be fair I have not investigated if Langchain also supports these types of vector retrieval methods). I want to use LangChain because of its functionality for developing more complex prompt templates (similarly I have not really investigated if llama_index supports this).\nMy goal is to ultimately evaluate how these different retrieval methods perform within the context of the application/chatbot. I know how to evaluate them with a separate evaluation questions file, but I would like to do things like compare the speed and humanness of responses

In [23]:
dspy_evaluator_kwargs = {
    "num_threads": 5
}

evaluator(query_writer, **dspy_evaluator_kwargs)

Average Metric: 11.03 / 20 (55.2%): 100%|██████████| 20/20 [00:16<00:00,  1.23it/s]

2025/08/13 20:27:28 INFO dspy.evaluate.evaluate: Average Metric: 11.033333333333333 / 20 (55.2%)


EvaluationResult(score=55.17, results=<list of 20 results>)

In [24]:
import dspy

import logging

# Simple setup for Jupyter
logging.basicConfig(level=logging.INFO, force=True)
logging.getLogger('dspy.teleprompt.gepa').setLevel(logging.INFO)
logging.getLogger('gepa').setLevel(logging.INFO)

# SILENCE the noisy HTTP loggers
logging.getLogger('httpx').setLevel(logging.WARNING)  # Only warnings and errors
logging.getLogger('openai').setLevel(logging.WARNING)
logging.getLogger('weaviate').setLevel(logging.WARNING)
logging.getLogger('httpcore').setLevel(logging.WARNING)

reflection_lm = dspy.LM(
    model="gpt-5",
    temperature=1.0,
    max_tokens=32_000
)

optimizer = dspy.GEPA(
    metric=metric_for_gepa,
    max_metric_calls=500,
    reflection_lm=reflection_lm,
    reflection_minibatch_size=5,
    use_merge=True,
    num_threads=8
)

# there are 30 samples in `trainset` to begin with
trainset=trainset[:15] # these are randomly sampled for Reflective Prompt Mutation
valset=trainset[15:] # these samples create the pareto frontier

optimized_query_expander = optimizer.compile(
    query_writer,
    trainset=trainset,
    valset=valset
)

2025/08/13 20:32:56 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 500 metric calls of the program. This amounts to 33.33 full evals on the train+val set.
2025/08/13 20:32:56 INFO dspy.teleprompt.gepa.gepa: Using 15 examples for tracking Pareto scores. You can consider using a sample of the valset to allow GEPA to explore more diverse solutions within the same budget.
2025/08/13 20:33:06 INFO dspy.evaluate.evaluate: Average Metric: 8.0 / 15 (53.3%)
2025/08/13 20:33:06 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 0.5333333333333333
2025/08/13 20:33:06 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Selected program 0 score: 0.5333333333333333


Average Metric: 3.83 / 5 (76.7%): 100%|██████████| 5/5 [00:04<00:00,  1.14it/s] 

2025/08/13 20:33:10 INFO dspy.evaluate.evaluate: Average Metric: 3.833333333333333 / 5 (76.7%)


2025/08/13 20:35:13 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Proposed new text for expand_query: You are given a user’s technical question. Your task is to expand it into a search-engine-optimized query that will help find authoritative answers, fixes, and examples.

How to write the expanded query:
- Preserve the user’s core ask, and add precise technical keywords: library/framework names, class/function/method names, parameters, config flags, environment variables, file/dir names, API endpoints, error messages, and exact version numbers when present.
- Include likely adjacent terms: “how to,” “best practices,” “known issues/bugs,” “workaround,” “version compatibility,” “examples,” “code snippets,” “configuration,” “persistence,” “performance,” “billing/costs,” “API semantics,” “syntax.”
- If the user provided code, extract and reference exact symbols (e.g., VectorstoreIndexCreator, RetrievalQA, as_retriever, JSONLoader, PromptTemplate, BaseCallbackHandler.on_llm_end, Redis simila

Average Metric: 2.83 / 5 (56.7%): 100%|██████████| 5/5 [00:04<00:00,  1.11it/s] 

...

2025/08/13 21:29:34 INFO dspy.evaluate.evaluate: Average Metric: 3.8333333333333335 / 5 (76.7%)


2025/08/13 21:30:24 INFO dspy.teleprompt.gepa.gepa: Iteration 36: Proposed new text for expand_query: You expand developer questions into a single, concise search-query paragraph that helps a search engine retrieve high-signal, authoritative fixes and examples.

Output format
- Return only the expanded search query as one paragraph (no headings or extra commentary), 2–5 sentences total.

How to expand
1) Extract exact technologies, libraries, versions, models, classes/functions, parameters, flags, and quote error messages verbatim. State the user’s goal, where it’s failing, environment details (Python version, package versions), and minimal repro hints.
2) Add synonyms/related names (e.g., LlamaIndex/llama_index/gpt_index; ChatOpenAI vs OpenAI wrappers; HuggingFacePipeline vs transformers.pipeline), and known breaking changes or package splits.
3) Anticipate root causes and fixes: version/compatibility issues, correct imports, supported tasks, parameters/flags, env vars, install comman

Average Metric: 4.50 / 5 (90.0%): 100%|██████████| 5/5 [00:08<00:00,  1.64s/it] 

2025/08/13 21:30:39 INFO dspy.evaluate.evaluate: Average Metric: 4.5 / 5 (90.0%)


2025/08/13 21:31:30 INFO dspy.teleprompt.gepa.gepa: Iteration 37: Proposed new text for expand_query: You expand developer questions into a single, concise search-query paragraph that helps a search engine retrieve high-signal, authoritative fixes and examples.

Output format
- Return only the expanded search query as one paragraph (no headings or extra commentary), 2–5 sentences total.

How to expand
1) Extract exact technologies, libraries, versions, models, classes/functions, parameters/flags, environment variables, and verbatim error messages. State the user’s task and where it fails.
2) Add synonyms and related names (e.g., LlamaIndex/llama_index/gpt_index; ChatOpenAI vs OpenAI wrappers; HuggingFacePipeline vs transformers.pipeline; ChatHuggingFace; OpenAI vs OpenAI-compatible server), plus common misconfigurations and breaking changes.
3) Anticipate root causes and fixes: version/compatibility issues, correct imports, supported tasks, proper parameters/flags, environment variable

In [25]:
print("GEPA run is finished!")

GEPA run is finished!


In [26]:
optimized_query_expander.save("gepa_optimized_query_expander.json")

In [28]:
evaluator(optimized_query_expander, **dspy_evaluator_kwargs)

Average Metric: 12.78 / 20 (63.9%): 100%|██████████| 20/20 [00:21<00:00,  1.06s/it]

2025/08/13 21:32:33 INFO dspy.evaluate.evaluate: Average Metric: 12.783333333333333 / 20 (63.9%)


EvaluationResult(score=63.92, results=<list of 20 results>)